In [391]:
from matplotlib.animation import FuncAnimation
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

plt.rcParams.update({'font.size': 16, 'figure.figsize': (8, 8)})
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",   # closest math font to Times
})

In [392]:
G   = 4 * np.pi**2   # AU^3 / (M_sun * yr^2)
M   = 1.0            # Solar mass
GM  = G * M
M_J = 9.548e-4       # Jupiter mass in solar masses
GM_J = G * M_J

In [393]:
def accel_voyager(xv, yv, xj, yj):
    # From Sun
    rv  = np.sqrt(xv**2 + yv**2)
    axs = -GM   * xv / rv**3
    ays = -GM   * yv / rv**3
    # From Jupiter
    dxj = xv - xj
    dyj = yv - yj
    rj  = np.sqrt(dxj**2 + dyj**2)
    axj = -GM_J * dxj / rj**3
    ayj = -GM_J * dyj / rj**3
    return axs + axj, ays + ayj

def accel_jupiter(xj, yj):
    rj  = np.sqrt(xj**2 + yj**2)
    ax = -GM * xj / rj**3
    ay = -GM * yj / rj**3
    return ax,ay

In [394]:
# Leapfrog for 3-body system 
def leapfrog_3body(xv0, yv0, vxv0, vyv0,#voyager initial values
                   xj0, yj0, vxj0, vyj0, #jupiter initial values
                   h, N):
    xv  = np.zeros(N+1);  yv  = np.zeros(N+1)
    vxv = np.zeros(N+1);  vyv = np.zeros(N+1)
    xj  = np.zeros(N+1);  yj  = np.zeros(N+1)
    vxj = np.zeros(N+1);  vyj = np.zeros(N+1)

    xv[0], yv[0]   = xv0, yv0
    vxv[0], vyv[0] = vxv0, vyv0
    xj[0], yj[0]   = xj0, yj0
    vxj[0], vyj[0] = vxj0, vyj0

    # Initial accelerations
    axv, ayv = accel_voyager(xv[0], yv[0], xj[0], yj[0])
    axj, ayj = accel_jupiter(xj[0], yj[0])

    for i in range(N):
       
        vxv_half = vxv[i] + 0.5 * axv * h
        vyv_half = vyv[i] + 0.5 * ayv * h
        vxj_half = vxj[i] + 0.5 * axj * h
        vyj_half = vyj[i] + 0.5 * ayj * h

        
        xv[i+1] = xv[i] + vxv_half * h
        yv[i+1] = yv[i] + vyv_half * h
        xj[i+1] = xj[i] + vxj_half * h
        yj[i+1] = yj[i] + vyj_half * h

        
        axv, ayv = accel_voyager(xv[i+1], yv[i+1], xj[i+1], yj[i+1])
        axj, ayj = accel_jupiter(xj[i+1], yj[i+1])

        
        vxv[i+1] = vxv_half + 0.5 * axv * h
        vyv[i+1] = vyv_half + 0.5 * ayv * h
        vxj[i+1] = vxj_half + 0.5 * axj * h
        vyj[i+1] = vyj_half + 0.5 * ayj * h

    return xv, yv, vxv, vyv, xj, yj, vxj, vyj

In [395]:
r_E = 1.0
r_J = 5.2
a_transfer = (r_E + r_J) / 2.0
T_h        = np.pi * np.sqrt(a_transfer**3 / GM)

v_H = np.sqrt(GM * (2/r_E - 1/a_transfer))

T_J     = r_J**1.5
omega_J = 2*np.pi / T_J
v_J     = np.sqrt(GM / r_J)

# Voyager on negative y-axis, velocity tangential (counterclockwise) = +x
xv0  =  0.0;  yv0  = -r_E
vxv0 =  v_H;  vyv0 =  0.0

# Encounter is now at (0, +r_J) — top of the plot — phi_enc = pi/2
# Jupiter walks backward from pi/2 by omega_J*T_h → lands in first quadrant
phi_J0 = np.pi/2 - omega_J * T_h + np.radians(4.0)
xj0  =  r_J * np.cos(phi_J0)
yj0  =  r_J * np.sin(phi_J0)
vxj0 = -v_J * np.sin(phi_J0)
vyj0 =  v_J * np.cos(phi_J0)

print(f"Transfer time   : {T_h:.3f} yr  ({T_h*12:.2f} months)")
print(f"Voyager launch  : ({xv0:.2f}, {yv0:.2f}) AU  |  v = ({vxv0:.3f}, {vyv0:.3f}) AU/yr")
print(f"Jupiter angle   : {np.degrees(phi_J0):.2f} deg")
print(f"Jupiter start   : ({xj0:.2f}, {yj0:.2f}) AU")
print(f"Encounter at    : (0.00, {r_J:.2f}) AU")

Transfer time   : 2.729 yr  (32.75 months)
Voyager launch  : (0.00, -1.00) AU  |  v = (8.138, 0.000) AU/yr
Jupiter angle   : 11.15 deg
Jupiter start   : (5.10, 1.01) AU
Encounter at    : (0.00, 5.20) AU


In [396]:
# Hohmann transfer orbit parameters
a_transfer = (rE + rJ) / 2          # semi-major axis = 3.1 AU

# Transfer time (half-period of the ellipse)
t_transfer = np.pi * np.sqrt(a_transfer**3 / GM)
print(f"Transfer time: {t_transfer:.3f} yr = {t_transfer*12:.1f} months")

# Voyager launch speed at perihelion (vis-viva equation)
v_perihelion = np.sqrt(GM * (2/rE - 1/a_transfer))
print(f"Voyager launch speed: {v_perihelion * 4.74047:.2f} km/s")

# Jupiter's angular velocity
T_J = 2 * np.pi * np.sqrt(rJ**3 / GM)
omega_J = 2 * np.pi / T_J

# Jupiter's initial angle: must arrive at angle π when Voyager does
theta_J0 = np.pi - omega_J * t_transfer
print(f"Jupiter initial angle: {np.degrees(theta_J0):.1f} degrees")


xv0, yv0   =  0.0, -rE          # Voyager on -y axis
vxv0, vyv0 =  v_perihelion, 0.0  # launches in +x direction
#xv0, yv0   = rE, 0.0
#vxv0, vyv0 = 0.0, v_perihelion

# Jupiter unchanged
phi = +0.05   # radians — positive = Jupiter arrives slightly early
              # → Voyager passes behind it → bigger boost
              # try values between 0.02 and 0.08

#theta_J0 = np.pi - omega_J * t_transfer + phi
theta_J0   =  np.pi - omega_J * t_transfer + phi - np.pi/2  # Jupiter in 1st quadrant

xj0  =  rJ * np.cos(theta_J0)
yj0  =  rJ * np.sin(theta_J0)
vj   =  np.sqrt(GM / rJ)
vxj0 = -vj * np.sin(theta_J0)
vyj0 =  vj * np.cos(theta_J0)


Transfer time: 2.729 yr = 32.7 months
Voyager launch speed: 38.58 km/s
Jupiter initial angle: 97.1 degrees


In [397]:
# Extend simulation time a bit past the flyby
h = 0.001          # yr
N = 4000           # 5 years total

In [398]:
xv, yv, vxv, vyv, xj, yj, vxj, vyj = leapfrog_3body(
    xv0, yv0, vxv0, vyv0,
    xj0, yj0, vxj0, vyj0,
    h, N
)

In [399]:
# 1. Is there even a close flyby?
dist_JV = np.sqrt((xv - xj)**2 + (yv - yj)**2)
idx_min = np.argmin(dist_JV)
print(f"Closest approach : {dist_JV[idx_min]:.4f} AU")
#print(f"Time of flyby    : {t[idx_min]*12:.2f} months")

# 2. Is there actually a speed boost?
window = 500   # check speed before and after flyby
print(f"Speed before flyby : {speed_v_kms[idx_min - window]:.2f} km/s")
print(f"Speed at flyby     : {speed_v_kms[idx_min]:.2f} km/s")
print(f"Speed after flyby  : {speed_v_kms[idx_min + window]:.2f} km/s")

Closest approach : 0.0031 AU
Speed before flyby : 8.71 km/s
Speed at flyby     : 7.21 km/s
Speed after flyby  : 17.03 km/s


In [400]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(-6, 6)
ax.set_ylim(-2, 12)
ax.set_aspect('equal')
ax.set_xlabel('x [AU]')
ax.set_ylabel('y [AU]')
ax.set_title('Voyager 2 Gravity Assist around Jupiter')

# Sun
sun = ax.scatter(0, 0, color='orange', s=200, label='Sun', zorder=5)

# Orbit lines
voyager_line, = ax.plot([], [], 'b-', lw=1)
jupiter_line, = ax.plot([], [], 'r-', lw=1)

# Jupiter's circular orbit (dashed)
theta_orbit = np.linspace(0, 2*np.pi, 500)
ax.plot(rJ * np.cos(theta_orbit), rJ * np.sin(theta_orbit), 'r--', lw=1, alpha=0.4,label= "Jupiter Orbit")

# Moving dots
voyager_dot, = ax.plot([], [], 'bo', markersize=5, label='Voyager 2', zorder=10)
jupiter_dot, = ax.plot([], [], 'ro', markersize=8, label='Jupiter', zorder=10)

# Info text
time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)
speed_text = ax.text(0.02, 0.90, '', transform=ax.transAxes)

ax.legend(loc='upper right',fontsize=12)

def init():
    voyager_line.set_data([], [])
    jupiter_line.set_data([], [])
    voyager_dot.set_data([], [])
    jupiter_dot.set_data([], [])
    time_text.set_text('')
    speed_text.set_text('')
    return voyager_line, jupiter_line, voyager_dot, jupiter_dot, time_text, speed_text

points_per_frame = 25  # controls speed of animation

def update(i):
    idx = i * points_per_frame
    if idx >= len(xv): idx = len(xv) - 1

    voyager_line.set_data(xv[:idx], yv[:idx])
    voyager_dot.set_data([xv[idx]], [yv[idx]])

    # jupiter_line removed — orbit already shown as dashed circle
    jupiter_dot.set_data([xj[idx]], [yj[idx]])

    time_text.set_text(f"Time = {idx*h*12:.1f} months")
    speed_text.set_text(f"v = {speed_v_kms[idx]:.2f} km/s")

    return voyager_line, jupiter_line, voyager_dot, jupiter_dot, time_text, speed_text

ani = animation.FuncAnimation(fig, update, frames=len(xv)//points_per_frame, 
                              init_func=init, blit=False, interval=50, repeat=False)

plt.close()
plt.tight_layout()
display(HTML(ani.to_html5_video()))

<Figure size 800x800 with 0 Axes>